Create the dataframe of anchors with embeddings

In [1]:
# Build the dataframe for anchors 

from FlagEmbedding import BGEM3FlagModel
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

from assets import anchors

# ---------------------------------------------------------
# 1. Inspect anchors.py & dynamically extract all uppercase string constants
# ---------------------------------------------------------
records = []
for attr in dir(anchors):
    if attr.isupper() and not attr.startswith("_"):
        val = getattr(anchors, attr)
        if isinstance(val, str) and val.strip():
            cat_name = attr.lower()
            records.append(
                {
                    "name": cat_name,
                    "label": f"is_{cat_name}",
                    "constant_name": attr,
                    "text": val.strip(),
                }
            )

# Create DataFrame sorted alphabetically by category name
df_anchors = (
    pd.DataFrame(records).sort_values("name").reset_index(drop=True)
)
print(
    f"Found {len(df_anchors)} anchor definitions in anchors.py: {df_anchors['name'].tolist()}"
)

# ---------------------------------------------------------
# 2. Initialize BGE-M3 Model on Mac MPS
# ---------------------------------------------------------
print("Loading BGE-M3 model on MPS...")
model = BGEM3FlagModel("../../Models/bge-m3", use_fp16=False, device="mps")
print("✅ BGE-M3 model loaded successfully.")

# ---------------------------------------------------------
# 3. Compute Dense Embeddings & L2-Normalize
# ---------------------------------------------------------
print(f"Encoding {len(df_anchors)} anchor definitions with BGE-M3...")
encodings = model.encode(
    df_anchors["text"].tolist(), return_dense=True, return_sparse=False
)

# Handle FlagEmbedding output dict vs raw array
if isinstance(encodings, dict) and "dense_vecs" in encodings:
    A_raw = np.array(encodings["dense_vecs"])
else:
    A_raw = np.array(encodings)

# Unit-normalize (||a||_2 = 1.0)
A_norm = normalize(A_raw, norm="l2", axis=1)
df_anchors["embedding"] = list(A_norm)

# ---------------------------------------------------------
# 4. Extract Matrix & Run Sanity Checks
# ---------------------------------------------------------
A_anchors = np.vstack(df_anchors["embedding"].to_numpy())
assert np.allclose(
    np.linalg.norm(A_anchors, axis=1), 1.0
), "Vectors are not unit-normalized!"

print(
    f"✅ Successfully built df_anchors: {A_anchors.shape[0]} categories x {A_anchors.shape[1]} dimensions"
)
display(
    df_anchors[["name", "label", "constant_name"]].assign(
        char_len=df_anchors["text"].str.len()
    )
)

Found 13 anchor definitions in anchors.py: ['afolu', 'buildings', 'cross_sectoral', 'cski', 'energy', 'ffoep', 'health', 'industry', 'oce', 'tfe', 'transport', 'urban', 'water']
Loading BGE-M3 model on MPS...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ BGE-M3 model loaded successfully.
Encoding 13 anchor definitions with BGE-M3...
✅ Successfully built df_anchors: 13 categories x 1024 dimensions


,name,label,constant_name,char_len
0,afolu,is_afolu,AFOLU,4969
1,buildings,is_buildings,BUILDINGS,3273
2,cross_sectoral,is_cross_sectoral,CROSS_SECTORAL,1744
3,cski,is_cski,CSKI,5830
4,energy,is_energy,ENERGY,6484
5,ffoep,is_ffoep,FFOEP,9285
6,health,is_health,HEALTH,5929
7,industry,is_industry,INDUSTRY,3818
8,oce,is_oce,OCE,4650
9,tfe,is_tfe,TFE,4420


In [2]:
# save on disk
df_anchors.to_parquet("assets/anchors.parquet")